# 1.查询保险产品及其条款文件

In [1]:
from app.infra.database import AsyncSessionFactory
from app.modules.product.service import ProductService

async with AsyncSessionFactory() as session:
    # 创建Service
    service = ProductService(session)
    # 查询
    products, total = await service.query_product_page(None, None, 1, 20)

# 打印
product = products[0]
print(product)

Product({'clause_name': '信美相互星辰守护B款少儿重疾险条款.pdf', 'insurer': '信美人寿相互保险社', 'name': '达尔文宝贝计划12号', 'description': '117种重疾+28种中症+45种轻症，少儿重疾保障灵活。', 'max_premium': Decimal('2501.83'), 'highlights': ['117种重疾', '少儿保障', '可选投保人豁免'], 'created_at': '2026-08-08 01:11:31', 'category': 'critical_illness', 'id': 1, 'image_url': 'https://files.huizecdn.com/file3/M00/91/12/rBYA3Gi1gZ2AXBL7AAYFJxRhbPg411.png', 'min_premium': Decimal('285.30'), 'target_group': '0-17周岁', 'status': 'active', 'updated_at': '2026-08-08 01:11:31'})


# 2.加载、解析条款文档

In [2]:
from mineru import MinerU
from app.core.config import settings

# 1.创建客户端
client = MinerU(settings.rag.mineru_token)
# 2.解析文件
result = client.extract(f"../../data/raw/kb/{product.clause_name}")

In [ ]:
# 三、文档优化
from langchain.chat_models import init_chat_model

MARKDOWN_OPTIMIZATION_PROMPT = """
下面这份Markdown文档是从保险条款PDF解析得来，由于PDF中的各个小节是以表格形式存在，所以解析时出现错乱。你分析内容，帮我转为格式正确的Markdown，特别是标题编号要正确。
- 标题等级要从1级标题开始，逐层增加，目录和文档名不计入标题等级。
- 输出结果中不要包含正文开始之前的部分。
- 输出结果只包含Markdown正文，不要解释或代码围栏。
- 不要修改保险条款。
""".strip()

llm = init_chat_model(
    "deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={'thinking': {'type': 'disabled'}},
    max_tokens=10000
)

response = llm.invoke([
    {"role": "system", "content": MARKDOWN_OPTIMIZATION_PROMPT},
    {"role": "user", "content": result.markdown},
])
markdown = response.content

# 3.切分文档

In [4]:
# 四、文档切分
from uuid import uuid4
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from app.rag.models import ParentChunk

# 1.准备TextSpliter
# 1.1.markdown切分器，切父块
markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=[
                ("#", "h1"),
                ("##", "h2"),
                ("###", "h3"),
                ("####", "h4"),
            ],
            strip_headers=True,
        )
# 1.2.递归切分器，切子块
child_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            separators=["\n\n", "\n", "。", "；", ";", "，", ","],
            keep_separator="end",
        )

# 2.先用markdown切分，得到父块小节信息
sections = markdown_splitter.split_text(markdown)

In [ ]:
# 3.处理父子块
parent_chunks: list[ParentChunk] = []
child_chunks: list[Document] = []

# 3.1.遍历父块小节
for section in sections:
    # 3.1.生成章节路径，section.metadata就是每一级标题的字典，values刚好得到标题列表
    section_path = list(section.metadata.values())
    # 3.2.组织 parent_chunk
    parent = ParentChunk(
        id=uuid4(),
        product_id=product.id,
        content=section.page_content,
        section_path=section_path,
        clause_name=product.clause_name
    )
    parent_chunks.append(parent)

    # 3.3.利用递归切分，切分父块，得到子块;
    # 重新组织section的metadata，会作为所有子块的metadata
    section.metadata = {
        "product_id": product.id,
        "parent_id": str(parent.id)
    }
    # 3.4.切分section，生成child的Document列表，并且沿用section的metadata
    children = child_splitter.split_documents([section])
    # 3.5.把父的section_path拼接到子content中，使信息完整
    section_path_header = "\n".join(section_path) + "\n"
    for c in children:
        c.page_content = section_path_header + c.page_content

    child_chunks.extend(children)

# 4.存入向量库、数据库

In [2]:
# 五、存入向量库
from langchain_ollama import OllamaEmbeddings
from langchain_milvus import Milvus, BM25BuiltInFunction
from app.core.config import settings

# 1.初始化向量模型
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

# 2、初始化 VectorStore，会自动创建Collection
vector_store = Milvus(
    embedding_function=embeddings,
    collection_name="insurance_collection",
    builtin_function=BM25BuiltInFunction(analyzer_params={"type": "chinese"}),
    vector_field=["dense", "sparse"],
    connection_args={"uri": settings.rag.milvus_url},
    auto_id=True,
)

# 3.添加新文档
vector_store.delete(expr=f"product_id == {product.id}")
vector_store.add_documents(child_chunks)

In [7]:
from app.rag.repository import ParentChunkRepository

async with AsyncSessionFactory() as session:
    async with session.begin():
        repo = ParentChunkRepository(session)
        # 1.先删除
        await repo.delete_by_product(product.id)
        # 2.再添加
        repo.add_all(parent_chunks)

In [3]:
# 测试Retriever
from app.rag.retriever import ParentChunkRetriever

retriever = ParentChunkRetriever(vector_store)

chunks = await retriever.retrieve("等待期是多久", 5, 1)